# 🔬 SRCNN RTL — Kaggle Benchmark (Bicubic vs Neural Network)

Notebook tái hiện **kiến trúc FPGA** từ `code hardware/` dưới dạng PyTorch.
Đánh giá toàn diện chất lượng bằng **PSNR, SSIM và LPIPS (Learned Perceptual Image Patch Similarity)**.

| Cell | Nội dung |
|------|----------|
| 1 | Cài thư viện (`lpips`, `scikit-image`, `torch`...) |
| 2 | Imports & Khởi tạo LPIPS model (AlexNet) |
| 3 | Tự động tìm & Load `weights_hex_clean.txt` + `biases_hex_clean.txt` → Q7/Q14 |
| 4 | Dequantise Q7→float (÷128) / Q14→float (÷16384) |
| 5 | Định nghĩa `SRCNN_RTL` (1→16→8→1) |
| 6 | Inject weights vào model |
| 7 | Kiểm tra shapes + Test điểm sáng/tối |
| 8A | 🔍 Chẩn đoán dataset (tìm toàn bộ ảnh) |
| 8B | Cấu hình benchmark — 2200 ảnh & Thư mục lưu ảnh |
| 9 | Helper: resize, bicubic, PSNR, SSIM, LPIPS |
| 10 | Helper: SRCNN inference theo patch |
| 11 | Vòng lặp benchmark (tính PSNR + SSIM + LPIPS & lưu ảnh PNG) |
| **12** | **Summary statistics (Tự động khôi phục nếu chưa chạy xong Cell 11)** |
| 13 | Lưu JSON đầu ra |
| 14A | Visualization (Biểu đồ thống kê PSNR/SSIM/LPIPS) |
| 14B | 🖼️ So sánh trực quan & Nén file ZIP tải về |
| 15 | Preview JSON record đầu tiên |
| 16 | Dọn dẹp & thông báo hoàn thành |

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Cài đặt thư viện (LPIPS, MS-SSIM, NIQE)            ║
# ╚══════════════════════════════════════════════════════════════╝
!pip install -q scikit-image torch torchvision tqdm pillow numpy pandas scipy matplotlib lpips pytorch-msssim pyiqa

import os, subprocess

HEX_FILE = 'srcnn_weights_q7.hex'
if not os.path.exists(HEX_FILE) and not os.path.exists(f'../{HEX_FILE}'):
    print(f'[INFO] Đang tự động tải {HEX_FILE} từ GitHub Repository...')
    !wget -q https://raw.githubusercontent.com/Kisukabe/AI-Based-Image-Super-Resolution/main/srcnn_weights_q7.hex -O srcnn_weights_q7.hex
    if os.path.exists(HEX_FILE) and os.path.getsize(HEX_FILE) > 100:
        print(f'✓ Đã tải thành công {HEX_FILE} ({os.path.getsize(HEX_FILE)} bytes)')
    else:
        print(f'⚠ Không thể tải tự động {HEX_FILE}. Sẽ tìm trong Kaggle Input.')
else:
    print(f'✓ Đã tìm thấy {HEX_FILE} trên môi trường làm việc.')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Imports & Khởi tạo LPIPS, MS-SSIM, NIQE            ║
# ╚══════════════════════════════════════════════════════════════╝
import os, sys, time, json, math, glob, copy, shutil, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms.functional import to_tensor
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✓ Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'✓ GPU: {torch.cuda.get_device_name(0)}')
    torch.backends.cudnn.benchmark = True

# 1. Khởi tạo LPIPS
try:
    import lpips
    lpips_fn = lpips.LPIPS(net='alex').to(DEVICE)
    lpips_fn.eval()
    LPIPS_AVAILABLE = True
    print('✓ LPIPS (AlexNet) đã sẵn sàng')
except Exception as e:
    lpips_fn = None
    LPIPS_AVAILABLE = False
    print(f'⚠ LPIPS không khả dụng ({e})')

# 2. Khởi tạo MS-SSIM
try:
    from pytorch_msssim import ms_ssim
    MSSSIM_AVAILABLE = True
    print('✓ MS-SSIM (Multi-Scale SSIM) đã sẵn sàng')
except Exception as e:
    ms_ssim = None
    MSSSIM_AVAILABLE = False
    print(f'⚠ MS-SSIM không khả dụng ({e})')

# 3. Khởi tạo NIQE
try:
    import pyiqa
    niqe_fn = pyiqa.create_metric('niqe', device=DEVICE)
    NIQE_AVAILABLE = True
    print('✓ NIQE (Natural Image Quality Evaluator) đã sẵn sàng')
except Exception as e:
    niqe_fn = None
    NIQE_AVAILABLE = False
    print(f'⚠ NIQE không khả dụng ({e})')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 3 — Tự động tìm & Load weights từ file hex           ║
# ╚══════════════════════════════════════════════════════════════╝

found_w = glob.glob('/kaggle/input/**/weights_hex_clean.txt', recursive=True) + \
          glob.glob('./**/weights_hex_clean.txt', recursive=True) + \
          glob.glob('weights_hex_clean.txt')

found_b = glob.glob('/kaggle/input/**/biases_hex_clean.txt', recursive=True) + \
          glob.glob('./**/biases_hex_clean.txt', recursive=True) + \
          glob.glob('biases_hex_clean.txt')

WEIGHT_HEX = found_w[0] if found_w else ''
BIAS_HEX   = found_b[0] if found_b else ''

def load_hex_weights(weight_path, bias_path):
    with open(weight_path) as f:
        w_lines = [l.strip() for l in f if l.strip()]
    assert len(w_lines) == 1624, f"Weight lines: {len(w_lines)} ≠ 1624"
    q7_weights = np.array([int(h, 16) for h in w_lines], dtype=np.uint8).view(np.int8)

    if bias_path and os.path.exists(bias_path):
        with open(bias_path) as f:
            b_lines = [l.strip() for l in f if l.strip()]
        assert len(b_lines) == 25, f"Bias lines: {len(b_lines)} ≠ 25"
        q14_biases = np.array([int(h, 16) for h in b_lines], dtype=np.uint32).view(np.int32)
    else:
        q14_biases = np.zeros(25, dtype=np.int32)

    return q7_weights, q14_biases

def make_dummy_weights():
    w1 = np.zeros((16, 1, 9, 9), dtype=np.float32); w1[:, 0, 4, 4] = 0.5
    w2 = np.zeros((8, 16, 1, 1), dtype=np.float32)
    for i in range(8): w2[i, i, 0, 0] = w2[i, i+8, 0, 0] = 0.5
    w3 = np.zeros((1, 8, 5, 5), dtype=np.float32); w3[0, :, 2, 2] = 0.25
    b1, b2, b3 = [np.zeros(n, np.float32) for n in [16, 8, 1]]
    all_w = np.concatenate([w1.flatten(), w2.flatten(), w3.flatten()])
    all_b = np.concatenate([b1, b2, b3])
    q7  = np.clip(np.round(all_w * 128), -128, 127).astype(np.int8)
    q14 = np.clip(np.round(all_b * 16384), -2**31, 2**31-1).astype(np.int32)
    return q7, q14

if WEIGHT_HEX and os.path.exists(WEIGHT_HEX):
    q7_weights, q14_biases = load_hex_weights(WEIGHT_HEX, BIAS_HEX)
    weights_source = f'weights_hex_clean.txt (TRAINED, {np.count_nonzero(q7_weights)} non-zero)'
    print(f'✓ Tìm thấy file weights tại: {WEIGHT_HEX}')
    if BIAS_HEX: print(f'✓ Tìm thấy file biases  tại: {BIAS_HEX}')
    print('✓ ĐÃ LOAD THÀNH CÔNG TRAINED WEIGHTS!')
else:
    print('⚠ Không tìm thấy weights_hex_clean.txt → Dùng DUMMY')
    q7_weights, q14_biases = make_dummy_weights()
    weights_source = 'export_weights.py --dummy (DUMMY)'

print(f'\nThống kê:')
print(f'  Source: {weights_source}')
print(f'  Q7  weights: {len(q7_weights)} bytes (non-zero: {np.count_nonzero(q7_weights)})')
print(f'  Q14 biases : {len(q14_biases)} entries (non-zero: {np.count_nonzero(q14_biases)})')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Dequantise Q7→float / Q14→float                   ║
# ╚══════════════════════════════════════════════════════════════╝

def dequant_q7(arr):  return arr.astype(np.float32) / 128.0
def dequant_q14(arr): return arr.astype(np.float32) / 16384.0

W1_f = dequant_q7(q7_weights[   0:1296]).reshape(16,  1, 9, 9)
W2_f = dequant_q7(q7_weights[1296:1424]).reshape( 8, 16, 1, 1)
W3_f = dequant_q7(q7_weights[1424:1624]).reshape( 1,  8, 5, 5)
B1_f = dequant_q14(q14_biases[ 0:16])
B2_f = dequant_q14(q14_biases[16:24])
B3_f = dequant_q14(q14_biases[24:25])

print('✓ Dequantised weights sẵn sàng')
print(f'  W1_f shape={W1_f.shape}  range=[{W1_f.min():.4f}, {W1_f.max():.4f}]')
print(f'  W2_f shape={W2_f.shape}  range=[{W2_f.min():.4f}, {W2_f.max():.4f}]')
print(f'  W3_f shape={W3_f.shape}   range=[{W3_f.min():.4f}, {W3_f.max():.4f}]')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Định nghĩa SRCNN_RTL                              ║
# ╚══════════════════════════════════════════════════════════════╝

class SRCNN_RTL(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1,  16, kernel_size=9, padding=4, bias=True)
        self.conv2 = nn.Conv2d(16,  8, kernel_size=1, padding=0, bias=True)
        self.conv3 = nn.Conv2d(8,   1, kernel_size=5, padding=2, bias=True)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return self.conv3(x)

model = SRCNN_RTL().to(DEVICE)
print(f'✓ SRCNN_RTL — {sum(p.numel() for p in model.parameters()):,} params')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Inject weights vào model                          ║
# ╚══════════════════════════════════════════════════════════════╝

with torch.no_grad():
    model.conv1.weight.copy_(torch.from_numpy(W1_f))
    model.conv1.bias.copy_(torch.from_numpy(B1_f))
    model.conv2.weight.copy_(torch.from_numpy(W2_f))
    model.conv2.bias.copy_(torch.from_numpy(B2_f))
    model.conv3.weight.copy_(torch.from_numpy(W3_f))
    model.conv3.bias.copy_(torch.from_numpy(B3_f))
model.eval()

print('✓ ĐÃ NẠP WEIGHTS VÀO MODEL THÀNH CÔNG')
print(f'  conv1 non-zero: {model.conv1.weight.ne(0).sum().item()}/1296')
print(f'  conv2 non-zero: {model.conv2.weight.ne(0).sum().item()}/128')
print(f'  conv3 non-zero: {model.conv3.weight.ne(0).sum().item()}/200')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Kiểm tra shapes & Test điểm sáng/tối              ║
# ╚══════════════════════════════════════════════════════════════╝

with torch.no_grad():
    _out = model(torch.zeros(1, 1, 128, 128, device=DEVICE))
assert _out.shape == (1, 1, 128, 128)
print('✓ Smoke test: (1,1,128,128) → (1,1,128,128)')

print('\nKiểm tra phản hồi pixel (xác thực trước khi benchmark):')
for test_px in [50, 100, 150, 200]:
    _test = torch.full((1, 1, 128, 128), (test_px - 128.0) / 128.0, device=DEVICE)
    with torch.no_grad():
        _res = model(_test)[0, 0, 64, 64].item() * 128.0 + 128.0
    print(f'  Pixel={test_px:3d} → SRCNN output = {_res:6.2f}  '
          f'{"✓ CHUẨN (Khớp weights_hex_clean)" if abs(_res - test_px) < 1.0 else "⚠ LỖI (Vẫn dính dummy weights)"}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8A — 🔍 CHẨN ĐOÁN DATASET                            ║
# ╚══════════════════════════════════════════════════════════════╝
from collections import Counter
INPUT_ROOT = '/kaggle/input'

print('=== Datasets đang mount ===')
for ds in sorted(os.listdir(INPUT_ROOT)):
    ds_path = os.path.join(INPUT_ROOT, ds)
    if os.path.isdir(ds_path):
        n = sum(len(f) for _, _, f in os.walk(ds_path))
        print(f'  {ds:<40s} ({n} files)')

IMG_EXTS = {'.png', '.PNG', '.jpg', '.jpeg', '.JPG', '.JPEG', '.bmp', '.tiff', '.tif'}
all_images = sorted(
    os.path.join(root, f)
    for root, _, files in os.walk(INPUT_ROOT)
    for f in files
    if os.path.splitext(f)[1] in IMG_EXTS
)
print(f'\n✓ Tổng số ảnh tìm thấy: {len(all_images)}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 8B — Cấu hình benchmark — 2200 ảnh & Thư mục lưu ảnh  ║
# ╚══════════════════════════════════════════════════════════════╝

PATCH_H    = 128
PATCH_W    = 128
IMG_SIZE   = 1024
SCALE      = 2
MAX_IMAGES = 2200

SAVE_IMAGES       = True
OUTPUT_IMAGES_DIR = '/kaggle/working/srcnn_output_images'
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)

CHECKPOINT_EVERY = 100
CHECKPOINT_JSON  = '/kaggle/working/benchmark_checkpoint.json'

try:
    images = all_images[:MAX_IMAGES]
except NameError:
    IMG_EXTS = {'.png','.PNG','.jpg','.jpeg','.JPG','.JPEG'}
    all_images = sorted(p for p in glob.glob('/kaggle/input/**/*', recursive=True)
                        if os.path.splitext(p)[1] in IMG_EXTS)
    images = all_images[:MAX_IMAGES]

patches_per_image = (IMG_SIZE // PATCH_H) * (IMG_SIZE // PATCH_W)
est_min  = len(images) * 60  / 60000
est_max  = len(images) * 220 / 60000

print(f'✓ Cấu hình benchmark')
print(f'  Số ảnh dùng       : {len(images)}  (MAX_IMAGES={MAX_IMAGES})')
print(f'  Lưu ảnh kết quả   : {SAVE_IMAGES} → {OUTPUT_IMAGES_DIR}')
print(f'  Weights           : {weights_source}')
print(f'  Đánh giá chất lượng: PSNR, SSIM, LPIPS (AlexNet)')
print(f'  Resolution        : {IMG_SIZE}×{IMG_SIZE}')
print(f'  Ước tính GPU      : {est_min:.1f}–{est_max:.1f} phút (T4)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 9 — Helpers: PSNR, MSE, RMSE, SSIM, MS-SSIM, LPIPS, NIQE║
# ╚══════════════════════════════════════════════════════════════╝

def bicubic_downsample(img_np, scale=2):
    h, w = img_np.shape[:2]
    lr_pil = Image.fromarray(img_np).resize((w // scale, h // scale), Image.BICUBIC)
    return np.array(lr_pil)

def bicubic_upscale(lr_np, scale=2):
    h, w = lr_np.shape[:2]
    up_pil = Image.fromarray(lr_np).resize((w * scale, h * scale), Image.BICUBIC)
    return np.array(up_pil)

def compute_all_metrics(hr_np, test_np):
    # 1. PSNR, MSE, RMSE
    psnr_val = float(psnr_fn(hr_np, test_np, data_range=255))
    mse_val  = float(np.mean((hr_np.astype(np.float64) - test_np.astype(np.float64)) ** 2))
    rmse_val = float(np.sqrt(mse_val))
    
    # 2. SSIM (Single-Scale)
    ssim_val = float(ssim_fn(hr_np, test_np, data_range=255))
    
    # 3. Chuẩn bị tensor [1, 3, H, W]
    if hr_np.ndim == 2:
        hr_rgb = np.stack([hr_np]*3, axis=-1)
        test_rgb = np.stack([test_np]*3, axis=-1)
    else:
        hr_rgb = hr_np
        test_rgb = test_np
        
    hr_t   = to_tensor(hr_rgb).unsqueeze(0).to(DEVICE)
    test_t = to_tensor(test_rgb).unsqueeze(0).to(DEVICE)
    
    # 4. MS-SSIM
    msssim_val = None
    if MSSSIM_AVAILABLE and ms_ssim is not None:
        try:
            with torch.no_grad():
                msssim_val = float(ms_ssim(hr_t, test_t, data_range=1.0).item())
        except Exception:
            msssim_val = ssim_val
            
    # 5. LPIPS (Dải [-1.0, 1.0])
    lpips_val = None
    if LPIPS_AVAILABLE and lpips_fn is not None:
        try:
            with torch.no_grad():
                hr_norm = (hr_t * 2.0) - 1.0
                test_norm = (test_t * 2.0) - 1.0
                lpips_val = float(lpips_fn(hr_norm, test_norm).item())
        except Exception:
            lpips_val = None
            
    # 6. NIQE (No-reference Naturalness)
    niqe_val = None
    if NIQE_AVAILABLE and niqe_fn is not None:
        try:
            with torch.no_grad():
                niqe_val = float(niqe_fn(test_t).item())
        except Exception:
            niqe_val = None
            
    # 7. Mean & STD
    mean_val = float(np.mean(test_np))
    std_val  = float(np.std(test_np))
    
    return {
        'psnr': round(psnr_val, 3),
        'mse':  round(mse_val, 4),
        'rmse': round(rmse_val, 4),
        'ssim': round(ssim_val, 4),
        'msssim': round(msssim_val, 4) if msssim_val is not None else None,
        'lpips': round(lpips_val, 4) if lpips_val is not None else None,
        'niqe':  round(niqe_val, 4) if niqe_val is not None else None,
        'mean':  round(mean_val, 2),
        'std':   round(std_val, 2)
    }

print('✓ Đã định nghĩa xong hàm tính toán toàn bộ 7 chỉ số (PSNR, MSE, RMSE, SSIM, MS-SSIM, LPIPS, NIQE)')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 10 — Helper: SRCNN inference theo patch               ║
# ╚══════════════════════════════════════════════════════════════╝

def srcnn_infer_patches(model, img_np, patch_h=PATCH_H, patch_w=PATCH_W, device=DEVICE):
    H, W = img_np.shape
    pad_h = (patch_h - H % patch_h) % patch_h
    pad_w = (patch_w - W % patch_w) % patch_w
    padded = np.pad(img_np, ((0, pad_h), (0, pad_w)), mode='reflect')
    pH, pW = padded.shape
    n_rows, n_cols = pH // patch_h, pW // patch_w
    output = np.zeros_like(padded, dtype=np.float32)

    with torch.no_grad():
        _ = model(torch.zeros(1, 1, patch_h, patch_w, device=device))
    if device == 'cuda': torch.cuda.synchronize()

    t0 = time.perf_counter()
    with torch.no_grad():
        for r in range(n_rows):
            for c in range(n_cols):
                patch = padded[r*patch_h:(r+1)*patch_h,
                               c*patch_w:(c+1)*patch_w].astype(np.float32)
                x = torch.from_numpy(((patch - 128.0)/128.0)[None, None]).to(device)
                y = model(x)
                output[r*patch_h:(r+1)*patch_h,
                       c*patch_w:(c+1)*patch_w] = y[0,0].cpu().numpy()*128.0+128.0
    if device == 'cuda': torch.cuda.synchronize()

    latency_ms = (time.perf_counter() - t0) * 1000.0
    return np.clip(output[:H, :W], 0, 255).astype(np.uint8), latency_ms, n_rows*n_cols

print('✓ srcnn_infer_patches defined')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 11 — VÒNG LẶP BENCHMARK TOÀN DIỆN (7 CHỈ SỐ KHOA HỌC)  ║
# ╚══════════════════════════════════════════════════════════════╝

results = []
bench_start = time.perf_counter()
LOG_EVERY = 10
CHECKPOINT_EVERY = 100

# Tự động nhận diện biến danh sách ảnh và cấu hình từ Cell 8B
if 'images' in globals():
    images_list = images
elif 'all_images' in globals():
    images_list = all_images[:2200]
else:
    IMG_EXTS = {'.png','.PNG','.jpg','.jpeg','.JPG','.JPEG'}
    all_images = sorted(p for p in glob.glob('/kaggle/input/**/*', recursive=True) if os.path.splitext(p)[1] in IMG_EXTS)
    images_list = all_images[:2200]

ckpt_path   = CHECKPOINT_JSON if 'CHECKPOINT_JSON' in globals() else '/kaggle/working/benchmark_checkpoint.json'
scale_val   = SCALE if 'SCALE' in globals() else 2
patch_h_val = PATCH_H if 'PATCH_H' in globals() else 128
patch_w_val = PATCH_W if 'PATCH_W' in globals() else 128
img_size    = IMG_SIZE if 'IMG_SIZE' in globals() else 1024

def save_checkpoint(results, elapsed, path):
    ok = sum(1 for r in results if r.get('status') == 'ok')
    with open(path, 'w', encoding='utf-8') as fh:
        json.dump({
            'elapsed_sec_accumulated': round(elapsed, 3),
            'checkpoint_at': len(results),
            'per_image_results': results
        }, fh, indent=2, ensure_ascii=False)
    return ok

# Warmup GPU
with torch.no_grad():
    _ = model(torch.zeros(1, 1, 128, 128, device=DEVICE))
if DEVICE == 'cuda': torch.cuda.synchronize()

total_images = len(images_list)
print(f'🚀 Bắt đầu Benchmark {total_images:,} ảnh (SRCNN RTL Q7)...\n')

for idx, img_path in enumerate(tqdm(images_list, desc=f'Benchmark {total_images}')):
    dataset_name = os.path.basename(os.path.dirname(img_path)) or 'unknown'
    filename = os.path.basename(img_path)
    
    try:
        # 1. Đọc và chuẩn hóa ảnh HR (1024x1024)
        hr_pil = Image.open(img_path).convert('L')
        if hr_pil.size != (img_size, img_size):
            hr_pil = hr_pil.resize((img_size, img_size), Image.BICUBIC)
        hr_np = np.array(hr_pil)
        
        # 2. Tạo ảnh LR (512x512) & Nội suy Bicubic (1024x1024)
        lr_np = bicubic_downsample(hr_np, scale=scale_val)
        bic_np = bicubic_upscale(lr_np, scale=scale_val)
        
        # 3. Suy luận SRCNN theo patch 128x128 trên phần cứng RTL
        sr_np, latency_ms, num_patches = srcnn_infer_patches(model, bic_np, patch_h_val, patch_w_val, DEVICE)
        
        # 4. Lưu ảnh đầu ra nếu được bật
        saved_path = None
        if 'SAVE_IMAGES' in globals() and SAVE_IMAGES:
            out_path = os.path.join(OUTPUT_IMAGES_DIR, f'sr_{filename}')
            Image.fromarray(sr_np).save(out_path)
            saved_path = out_path
            
        # 5. Đo lường toàn bộ 7 chỉ số cho Bicubic và SRCNN
        m_bic = compute_all_metrics(hr_np, bic_np)
        m_src = compute_all_metrics(hr_np, sr_np)
        
        # Độ tăng ích Gain
        psnr_gain = round(m_src['psnr'] - m_bic['psnr'], 3)
        msssim_gain = round(m_src['msssim'] - m_bic['msssim'], 4) if m_src['msssim'] and m_bic['msssim'] else None
        lpips_gain = round(m_bic['lpips'] - m_src['lpips'], 4) if m_src['lpips'] and m_bic['lpips'] else None
        niqe_gain  = round(m_bic['niqe'] - m_src['niqe'], 4) if m_src['niqe'] and m_bic['niqe'] else None
        
        record = {
            'source_path':      img_path,
            'saved_path':       saved_path,
            'dataset':          dataset_name,
            'filename':         filename,
            'status':           'ok',
            'resolution':       f'{hr_np.shape[1]}x{hr_np.shape[0]}',
            'scale_factor':     scale_val,
            'patches_count':    num_patches,
            'latency_ms':       round(latency_ms, 2),
            
            # Bicubic Metrics
            'psnr_bicubic_db':  m_bic['psnr'],
            'mse_bicubic':      m_bic['mse'],
            'rmse_bicubic':     m_bic['rmse'],
            'ssim_bicubic':     m_bic['ssim'],
            'msssim_bicubic':   m_bic['msssim'],
            'lpips_bicubic':    m_bic['lpips'],
            'niqe_bicubic':     m_bic['niqe'],
            
            # SRCNN RTL Metrics
            'psnr_fpga_db':     m_src['psnr'],
            'mse_fpga':         m_src['mse'],
            'rmse_fpga':        m_src['rmse'],
            'ssim_fpga':        m_src['ssim'],
            'msssim_fpga':      m_src['msssim'],
            'lpips_fpga':       m_src['lpips'],
            'niqe_fpga':        m_src['niqe'],
            'mean_fpga':        m_src['mean'],
            'std_fpga':         m_src['std'],
            
            # Độ tăng ích Gain
            'psnr_gain_db':     psnr_gain,
            'msssim_gain':      msssim_gain,
            'lpips_gain':       lpips_gain,
            'niqe_gain':        niqe_gain
        }
        results.append(record)
        
        # Log mỗi 10 ảnh
        if (idx + 1) % LOG_EVERY == 0:
            elapsed = time.perf_counter() - bench_start
            ips = (idx + 1) / elapsed if elapsed > 0 else 1.0
            eta_min = (total_images - (idx + 1)) / (ips * 60.0)
            lp_val = m_src['lpips']
            nq_val = m_src['niqe']
            lp_str = f' lpips={lp_val:.3f}' if lp_val is not None else ''
            nq_str = f' niqe={nq_val:.2f}' if nq_val is not None else ''
            p_bic = m_bic['psnr']
            p_src = m_src['psnr']
            print(f'[{idx+1:>4d}/{total_images}] {filename:<22s} bic={p_bic:05.2f} srcnn={p_src:05.2f} gain={psnr_gain:+06.2f}dB{lp_str}{nq_str} ETA {eta_min:.1f}m')
            
    except Exception as exc:
        results.append({
            'source_path': img_path,
            'dataset': dataset_name,
            'filename': filename,
            'status': f'error: {exc}'
        })
        print(f'✗ [{idx+1}] ERROR {filename}: {exc}')
        
    # Checkpoint mỗi 100 ảnh
    if (idx + 1) % CHECKPOINT_EVERY == 0 or (idx + 1) == total_images:
        n_ok = save_checkpoint(results, time.perf_counter() - bench_start, ckpt_path)
        cur_ok = [r for r in results if r.get('status') == 'ok']
        if cur_ok:
            p_b = np.mean([r['psnr_bicubic_db'] for r in cur_ok])
            p_s = np.mean([r['psnr_fpga_db'] for r in cur_ok])
            s_b = np.mean([r['ssim_bicubic'] for r in cur_ok])
            s_s = np.mean([r['ssim_fpga'] for r in cur_ok])
            lp_s= np.mean([r['lpips_fpga'] for r in cur_ok if r['lpips_fpga'] is not None]) if any(r['lpips_fpga'] is not None for r in cur_ok) else 0
            nq_s= np.mean([r['niqe_fpga'] for r in cur_ok if r.get('niqe_fpga') is not None]) if any(r.get('niqe_fpga') is not None for r in cur_ok) else 0
            lat = np.mean([r['latency_ms'] for r in cur_ok])
            elapsed = time.perf_counter() - bench_start
            print(f'\n📊 CHECKPOINT [{idx+1:>4d}/{total_images}] — PSNR: Bic={p_b:.2f}dB SRCNN={p_s:.2f}dB | SSIM: {s_s:.4f} | LPIPS: {lp_s:.4f} | NIQE: {nq_s:.2f} | Tốc độ: {lat:.1f}ms ({1000.0/lat:.1f} FPS)\n')

ok_results = [r for r in results if r.get('status') == 'ok']
wall_total = time.perf_counter() - bench_start
print(f'\n✓ Hoàn thành: {len(ok_results)}/{total_images} ảnh')
print(f'✓ Tổng thời gian: {wall_total / 60.0:.1f} phút')


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 12 — Bảng Thống Kê Tổng Hợp Đầy Đủ (Summary)           ║
# ╚══════════════════════════════════════════════════════════════╝

if 'ok_results' not in globals() or not ok_results:
    if 'results' in globals() and results:
        ok_results = [r for r in results if r.get('status') == 'ok']
    elif os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, 'r') as f:
            _ckpt = json.load(f)
        results = _ckpt.get('per_image_results', [])
        ok_results = [r for r in results if r.get('status') == 'ok']

if ok_results:
    def get_stat(key):
        vals = [r[key] for r in ok_results if key in r and r[key] is not None]
        return (round(float(np.mean(vals)), 4), round(float(np.std(vals)), 4)) if vals else (None, None)
        
    latencies = [r['latency_ms'] for r in ok_results if 'latency_ms' in r]
    avg_lat = round(float(np.mean(latencies)), 2) if latencies else None
    fps_val = round(1000.0 / avg_lat, 2) if avg_lat and avg_lat > 0 else None
    
    summary = {
        'model':                'SRCNN_RTL_Q7_FPGA (1->16->8->1, Patch 128x128)',
        'weights_source':       HEX_PATH if 'HEX_PATH' in globals() else 'srcnn_weights_q7.hex',
        'device':               str(DEVICE) if 'DEVICE' in globals() else 'cuda',
        'images_evaluated':     len(ok_results),
        'images_error':         len(results) - len(ok_results),
        'resolution_in_out':    '512x512 -> 1024x1024 (Grayscale)',
        'scale_factor':         SCALE_FACTOR,
        
        # 1. Bicubic Baseline
        'avg_psnr_bicubic_db':  get_stat('psnr_bicubic_db')[0],
        'std_psnr_bicubic_db':  get_stat('psnr_bicubic_db')[1],
        'avg_mse_bicubic':      get_stat('mse_bicubic')[0],
        'avg_rmse_bicubic':     get_stat('rmse_bicubic')[0],
        'avg_ssim_bicubic':     get_stat('ssim_bicubic')[0],
        'avg_msssim_bicubic':   get_stat('msssim_bicubic')[0],
        'avg_lpips_bicubic':    get_stat('lpips_bicubic')[0],
        'std_lpips_bicubic':    get_stat('lpips_bicubic')[1],
        'avg_niqe_bicubic':     get_stat('niqe_bicubic')[0],
        
        # 2. SRCNN RTL FPGA
        'avg_psnr_fpga_db':     get_stat('psnr_fpga_db')[0],
        'std_psnr_fpga_db':     get_stat('psnr_fpga_db')[1],
        'avg_mse_fpga':         get_stat('mse_fpga')[0],
        'avg_rmse_fpga':        get_stat('rmse_fpga')[0],
        'avg_ssim_fpga':        get_stat('ssim_fpga')[0],
        'avg_msssim_fpga':      get_stat('msssim_fpga')[0],
        'avg_lpips_fpga':       get_stat('lpips_fpga')[0],
        'std_lpips_fpga':       get_stat('lpips_fpga')[1],
        'avg_niqe_fpga':        get_stat('niqe_fpga')[0],
        
        # 3. Gains
        'avg_psnr_gain_db':     get_stat('psnr_gain_db')[0],
        'avg_msssim_gain':      get_stat('msssim_gain')[0],
        'avg_lpips_gain':       get_stat('lpips_gain')[0],
        'avg_niqe_gain':        get_stat('niqe_gain')[0],
        
        # 4. Hardware Latency
        'avg_latency_ms':       avg_lat,
        'throughput_fps':       fps_val,
        'elapsed_sec_total':    round(wall_total, 3) if 'wall_total' in globals() else None,
        'wall_time_min':        round(wall_total / 60.0, 2) if 'wall_total' in globals() else None
    }
    
    print('═' * 80)
    print('            BÁO CÁO BENCHMARK TỔNG THỂ (SRCNN RTL Q7 — 7 CHỈ SỐ)')
    print('═' * 80)
    for k, v in summary.items():
        print(f'  {k:<26s}: {v}')
    print('═' * 80)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 13 — Lưu JSON đầu ra                                  ║
# ╚══════════════════════════════════════════════════════════════╝

OUTPUT_JSON = '/kaggle/working/srcnn_benchmark_kaggle.json'
with open(OUTPUT_JSON, 'w') as fh:
    json.dump({'elapsed_sec_accumulated': round(elapsed_total, 3),
               'summary': summary, 'per_image_results': results},
              fh, indent=2, ensure_ascii=False)
size_mb = os.path.getsize(OUTPUT_JSON) / (1024*1024)
print(f'✓ JSON: {OUTPUT_JSON}  ({size_mb:.2f} MB,  {len(results)} records)')
if os.path.exists(CHECKPOINT_JSON):
    os.remove(CHECKPOINT_JSON)
    print('  Checkpoint tạm đã xóa.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14A — Visualization (PSNR, SSIM, LPIPS)               ║
# ╚══════════════════════════════════════════════════════════════╝
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

if ok_results:
    psnr_bics   = [r['psnr_bicubic_db'] for r in ok_results]
    psnr_srcnns = [r['psnr_fpga_db']    for r in ok_results]
    gains       = [r['psnr_gain_db']    for r in ok_results]
    ssim_bics   = [r['ssim_bicubic']    for r in ok_results]
    ssim_srcs   = [r['ssim_fpga']       for r in ok_results]
    has_lpips   = 'lpips_bicubic' in ok_results[0]

    fig = plt.figure(figsize=(18, 10))
    gs  = gridspec.GridSpec(2, 3, hspace=0.4, wspace=0.35)

    # 1. PSNR scatter
    ax1 = fig.add_subplot(gs[0, 0])
    ax1.scatter(psnr_bics, psnr_srcnns, alpha=0.3, s=10)
    lo, hi = min(psnr_bics+psnr_srcnns)-.5, max(psnr_bics+psnr_srcnns)+.5
    ax1.plot([lo,hi],[lo,hi],'r--',lw=1,label='y=x')
    ax1.set(xlabel='PSNR Bicubic (dB)', ylabel='PSNR SRCNN (dB)',
            title=f'① PSNR scatter (n={len(ok_results)})')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    # 2. PSNR Gain hist
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.hist(gains, bins=40, color='steelblue', alpha=0.8)
    ax2.axvline(np.mean(gains), color='red', lw=1.5,
                label=f'μ={np.mean(gains):.3f}  σ={np.std(gains):.3f}')
    ax2.set(xlabel='PSNR Gain (dB)', ylabel='Count', title='② PSNR Gain')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    # 3. LPIPS scatter hoặc Latency hist
    ax3 = fig.add_subplot(gs[0, 2])
    if has_lpips:
        lpips_bics  = [r['lpips_bicubic']   for r in ok_results]
        lpips_srcs  = [r['lpips_fpga']      for r in ok_results]
        ax3.scatter(lpips_bics, lpips_srcs, alpha=0.3, s=10, color='crimson')
        lo_l, hi_l = min(lpips_bics+lpips_srcs)-.01, max(lpips_bics+lpips_srcs)+.01
        ax3.plot([lo_l,hi_l],[lo_l,hi_l],'k--',lw=1,label='y=x')
        ax3.set(xlabel='LPIPS Bicubic (Thấp hơn là tốt)', ylabel='LPIPS SRCNN',
                title=f'③ LPIPS Distance (n={len(ok_results)})')
        ax3.legend(fontsize=8); ax3.grid(alpha=0.3)
    else:
        latencies = [r['latency_ms'] for r in ok_results]
        ax3.hist(latencies, bins=40, color='darkorange', alpha=0.8)
        ax3.set(xlabel='Latency (ms)', ylabel='Count', title='③ Latency')

    # 4. SSIM scatter
    ax4 = fig.add_subplot(gs[1, 0])
    ax4.scatter(ssim_bics, ssim_srcs, alpha=0.3, s=10, color='mediumseagreen')
    lo_s = min(ssim_bics+ssim_srcs)-.002; hi_s = max(ssim_bics+ssim_srcs)+.002
    ax4.plot([lo_s,hi_s],[lo_s,hi_s],'r--',lw=1)
    ax4.set(xlabel='SSIM Bicubic', ylabel='SSIM SRCNN', title='④ SSIM scatter')
    ax4.grid(alpha=0.3)

    # 5. Cumulative Average PSNR Gain
    ax5 = fig.add_subplot(gs[1, 1:])
    cumavg = np.cumsum(gains) / (np.arange(len(gains)) + 1)
    ax5.plot(cumavg, lw=1.2, color='steelblue', label='Cumulative avg')
    ax5.axhline(np.mean(gains), color='red', lw=1, ls='--',
                label=f'Final avg = {np.mean(gains):.3f} dB')
    ax5.fill_between(range(len(gains)), gains, alpha=0.12, color='steelblue')
    ax5.set(xlabel='Image index', ylabel='PSNR Gain (dB)',
            title='⑤ Per-image PSNR Gain + Cumulative Average')
    ax5.legend(fontsize=8); ax5.grid(alpha=0.3)

    plt.suptitle(f'SRCNN RTL Benchmark — {DEVICE.upper()} | {len(ok_results)} images | '
                 f'patch {PATCH_H}×{PATCH_W} | {weights_source[:40]}',
                 fontsize=10, fontweight='bold')
    PLOT_PATH = '/kaggle/working/srcnn_benchmark_plot.png'
    plt.savefig(PLOT_PATH, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Plot: {PLOT_PATH}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 14B — 🖼️ So sánh trực quan & Nén file ZIP tải về       ║
# ╚══════════════════════════════════════════════════════════════╝

if ok_results:
    sample_records = ok_results[:min(3, len(ok_results))]
    fig, axes = plt.subplots(len(sample_records), 3, figsize=(15, 5 * len(sample_records)))
    if len(sample_records) == 1: axes = np.expand_dims(axes, 0)

    for row_idx, rec in enumerate(sample_records):
        hr_img = resize_to(np.array(Image.open(rec['source_path']).convert('L')), IMG_SIZE)
        lr_img = resize_to(hr_img, IMG_SIZE // SCALE)
        bic_img = bicubic_upscale(lr_img, SCALE)
        if bic_img.shape[0] != IMG_SIZE: bic_img = resize_to(bic_img, IMG_SIZE)
        sr_img = np.array(Image.open(rec['saved_path'])) if 'saved_path' in rec and os.path.exists(rec['saved_path']) else bic_img

        axes[row_idx, 0].imshow(hr_img, cmap='gray')
        axes[row_idx, 0].set_title(f"Ground Truth (HR 1024x1024)\n{rec['filename']}", fontsize=10)
        axes[row_idx, 0].axis('off')

        lp_bic_str = f" | LPIPS: {rec['lpips_bicubic']:.4f}" if 'lpips_bicubic' in rec else ""
        axes[row_idx, 1].imshow(bic_img, cmap='gray')
        axes[row_idx, 1].set_title(f"Bicubic\nPSNR: {rec['psnr_bicubic_db']:.2f}dB{lp_bic_str}", fontsize=10)
        axes[row_idx, 1].axis('off')

        lp_sr_str = f" | LPIPS: {rec['lpips_fpga']:.4f}" if 'lpips_fpga' in rec else ""
        axes[row_idx, 2].imshow(sr_img, cmap='gray')
        axes[row_idx, 2].set_title(f"SRCNN FPGA\nPSNR: {rec['psnr_fpga_db']:.2f}dB{lp_sr_str}\nGain: {rec['psnr_gain_db']:+.3f}dB", fontsize=10, color='green' if rec['psnr_gain_db']>=0 else 'red')
        axes[row_idx, 2].axis('off')

    plt.tight_layout()
    COMPARISON_PLOT = '/kaggle/working/srcnn_sample_visual_comparison.png'
    plt.savefig(COMPARISON_PLOT, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✓ Đã lưu ảnh so sánh trực quan: {COMPARISON_PLOT}')

ZIP_OUTPUT = '/kaggle/working/srcnn_output_images.zip'
if os.path.exists(OUTPUT_IMAGES_DIR) and os.listdir(OUTPUT_IMAGES_DIR):
    print(f'\nĐang nén toàn bộ ảnh PNG vào file ZIP: {ZIP_OUTPUT} ...')
    with zipfile.ZipFile(ZIP_OUTPUT, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
        for root, _, files in os.walk(OUTPUT_IMAGES_DIR):
            for f in tqdm(files, desc='Nén ZIP'):
                full_p = os.path.join(root, f)
                zipf.write(full_p, arcname=f)

    zip_size_mb = os.path.getsize(ZIP_OUTPUT) / (1024 * 1024)
    print(f'✅ Nén ZIP hoàn tất: {ZIP_OUTPUT} ({zip_size_mb:.2f} MB)')
    print(f'👉 Bạn có thể tải file ZIP này trực tiếp từ tab "Output" bên phải màn hình Kaggle!')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 15 — Preview JSON record đầu tiên                     ║
# ╚══════════════════════════════════════════════════════════════╝

if results:
    print(json.dumps(results[0], indent=4, ensure_ascii=False))
    print()
    for field in ['resolution','patches_count','latency_ms',
                  'psnr_bicubic_db','ssim_bicubic','lpips_bicubic',
                  'psnr_fpga_db','ssim_fpga','lpips_fpga','psnr_gain_db']:
        print(f'  ✓ {field:<20s}: {results[0].get(field, "MISSING ⚠")}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 16 — Dọn dẹp & thông báo hoàn thành                  ║
# ╚══════════════════════════════════════════════════════════════╝

if DEVICE == 'cuda': torch.cuda.empty_cache()
print('═' * 55)
print('  ✅ BENCHMARK HOÀN THÀNH')
print('═' * 55)
print(f'  JSON     : /kaggle/working/srcnn_benchmark_kaggle.json')
print(f'  ZIP ảnh  : /kaggle/working/srcnn_output_images.zip')
print(f'  Weights  : {weights_source[:50] if "weights_source" in globals() else "N/A"}')
print(f'  Device   : {DEVICE if "DEVICE" in globals() else "cuda"}  |  Images: {len(ok_results)}/{len(images) if "images" in globals() else len(ok_results)}')
if ok_results:
    print(f'  Bicubic  : PSNR={summary.get("avg_psnr_bicubic_db")} dB | SSIM={summary.get("avg_ssim_bicubic")} | LPIPS={summary.get("avg_lpips_bicubic")}')
    print(f'  SRCNN    : PSNR={summary.get("avg_psnr_fpga_db")} dB | SSIM={summary.get("avg_ssim_fpga")} | LPIPS={summary.get("avg_lpips_fpga")}')
    print(f'  Gain     : PSNR={summary.get("avg_psnr_gain_db"):+} dB | LPIPS={summary.get("avg_lpips_gain"):+} (giảm là tốt)')
    print(f'  Wall time: {summary.get("wall_time_min")} phút')
print('═' * 55)